In [ ]:
import os
import sys
import json
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple
from datetime import datetime
import pandas as pd
from sklearn.ensemble import RandomForestRegressor

def obter_diretorio_input() -> Path:
    if "INPUT_DIR" in os.environ and os.path.isdir(os.environ["INPUT_DIR"]):
        return Path(os.environ["INPUT_DIR"])
    for p in [Path.cwd(), Path.cwd().parent, Path.cwd().parents[1]]:
        if (p / "workflows").is_dir():
            return p / "workflows"
        if (p / "hydranomic" / "workflows").is_dir():
            return p / "hydranomic" / "workflows"
    return Path.cwd()

def obter_diretorio_output() -> Path:
    if "OUTPUT_DIR" in os.environ:
        caminho = Path(os.environ["OUTPUT_DIR"])
        caminho.mkdir(parents=True, exist_ok=True)
        return caminho
    for p in [Path.cwd(), Path.cwd().parent, Path.cwd().parents[1]]:
        if (p / "workflows").is_dir():
            return p / "workflows"
        if (p / "hydranomic" / "workflows").is_dir():
            return p / "hydranomic" / "workflows"
    caminho = Path.cwd() / "outputs"
    caminho.mkdir(parents=True, exist_ok=True)
    return caminho

def carregar_json(nome_arquivo: str, padrao: Any = None) -> Any:
    dir_in = obter_diretorio_input()
    candidatos = [
        dir_in / nome_arquivo,
        Path.cwd() / "workflows" / nome_arquivo,
        Path.cwd() / "hydranomic" / "workflows" / nome_arquivo,
        Path.cwd() / nome_arquivo
    ]
    for c in candidatos:
        if c.exists() and c.stat().st_size > 0:
            with open(c, "r", encoding="utf-8") as f:
                return json.load(f)
    if padrao is not None:
        return padrao
    raise FileNotFoundError(f"Arquivo {nome_arquivo} nao encontrado em {dir_in}")

def salvar_json(nome_arquivo: str, dados: Any, indent: int = 2) -> Path:
    dir_out = obter_diretorio_output()
    caminho = dir_out / nome_arquivo
    with open(caminho, "w", encoding="utf-8") as f:
        json.dump(dados, f, ensure_ascii=False, indent=indent)
    return caminho

def carregar_contexto() -> Dict[str, Any]:
    try:
        return carregar_json("context.json", padrao={})
    except Exception:
        return {}


In [ ]:
ESTADO = "tudo"
TARIFA_AGUA = 59.6
ARQUIVO_HISTORICO = "conta_povoada.json"
N_ESTIMATORS = 250
ANO_GRAFICO = None


In [ ]:
def processar_dados_grafico(dados: Optional[List[Dict[str, Any]]] = None, ano_referencia: Optional[int] = None, salvar: bool = True) -> Dict[str, Any]:
    contexto = carregar_contexto()
    if dados is None:
        try:
            dados = carregar_json(ARQUIVO_HISTORICO)
        except Exception:
            dados = carregar_json("conta.json", padrao=[])
    if ano_referencia is None:
        if "ano_referencia" in contexto:
            ano_referencia = int(contexto["ano_referencia"])
        elif ANO_GRAFICO is not None:
            ano_referencia = int(ANO_GRAFICO)
        elif dados:
            anos_encontrados = []
            for r in dados:
                partes = str(r.get("id", "")).split("-")
                if len(partes) == 2 and partes[0].isdigit():
                    anos_encontrados.append(int(partes[0]))
            ano_referencia = max(anos_encontrados) if anos_encontrados else datetime.now().year
        else:
            ano_referencia = datetime.now().year
    consumo_mensal: Dict[int, Optional[float]] = {}
    mes_atual: Optional[int] = None
    for registro in dados:
        registro_id = str(registro.get("id", ""))
        separar_partes = registro_id.split("-")
        if len(separar_partes) != 2:
            continue
        try:
            ano_registro = int(separar_partes[0])
            mes_registro = int(separar_partes[1])
        except ValueError:
            continue
        if ano_registro == ano_referencia and 1 <= mes_registro <= 12:
            consumo_mensal[mes_registro] = registro.get("consumo_m3")
            mes_atual = mes_registro
    meses_tags = ["Jan", "Fev", "Mar", "Abr", "Mai", "Jun", "Jul", "Ago", "Set", "Out", "Nov", "Dez"]
    resultado = {
        "ano_referencia": ano_referencia,
        "mes_atual": mes_atual,
        "unidade_medida": "m³",
        "meses": [
            {
                "mes": mes,
                "rotulo": meses_tags[mes - 1],
                "consumo_m3": consumo_mensal.get(mes)
            }
            for mes in range(1, 13)
        ]
    }
    if salvar:
        salvar_json("grafico_anual.json", resultado)
    return resultado


In [ ]:
def executar_pipeline_ml(dados: Optional[List[Dict[str, Any]]] = None, tarifa: Optional[float] = None, salvar: bool = True) -> Dict[str, Any]:
    contexto = carregar_contexto()
    if tarifa is None:
        tarifa = float(contexto.get("tarifa", TARIFA_AGUA))
    if dados is None:
        try:
            dados = carregar_json(ARQUIVO_HISTORICO)
        except Exception:
            dados = carregar_json("conta.json", padrao=[])
    if not dados:
        raise ValueError("Dados insuficientes para treino do modelo de Machine Learning.")
    df = pd.DataFrame(dados)
    df = df.sort_values(by=["unidade_consumidora", "id"]).reset_index(drop=True)
    df["ano"] = df["id"].astype(str).str.split("-").str[0].astype(int)
    df["mes"] = df["id"].astype(str).str.split("-").str[1].astype(int)
    ano_inicial = df["ano"].min()
    mes_inicial = df.loc[df["ano"] == ano_inicial, "mes"].min()
    df["tempo_iniciofinal"] = (df["ano"] - ano_inicial) * 12 + (df["mes"] - mes_inicial)
    df["memoria_consumo"] = df.groupby("unidade_consumidora")["consumo_m3"].shift(1)
    df_treino = df.dropna(subset=["memoria_consumo"]).copy()
    features_entrada = ["mes", "tempo_iniciofinal", "memoria_consumo"]
    variavel_saida = "consumo_m3"
    modelo = RandomForestRegressor(n_estimators=N_ESTIMATORS, random_state=45)
    modelo.fit(df_treino[features_entrada], df_treino[variavel_saida])
    ultima_linha = df.iloc[-1]
    ultimo_ano = int(ultima_linha["ano"])
    ultimo_mes = int(ultima_linha["mes"])
    ultimo_consumo = float(ultima_linha["consumo_m3"])
    ultimo_passo = int(ultima_linha["tempo_iniciofinal"])
    if ultimo_mes == 12:
        proximo_mes = 1
        proximo_ano = ultimo_ano + 1
    else:
        proximo_mes = ultimo_mes + 1
        proximo_ano = ultimo_ano
    proximo_passo = ultimo_passo + 1
    dados_previsao = pd.DataFrame([[proximo_mes, proximo_passo, ultimo_consumo]], columns=features_entrada)
    previsao_m3 = float(modelo.predict(dados_previsao)[0])
    valor_conta = round(previsao_m3 * tarifa, 2)
    resultado = {
        "ano": proximo_ano,
        "mes": proximo_mes,
        "unidade_consumidora": int(ultima_linha["unidade_consumidora"]),
        "ultimo_consumo_registrado_m3": ultimo_consumo,
        "previsao_m3": round(previsao_m3, 2),
        "tarifa_m3": tarifa,
        "valor_estimado_conta": valor_conta,
        "modelo": "RandomForestRegressor"
    }
    if salvar:
        salvar_json("previsao_ml.json", resultado)
    return resultado


In [ ]:
def avaliar_progresso_meta(consumo_atual_m3: Optional[float] = None, salvar: bool = True) -> Dict[str, Any]:
    config = carregar_json("config.json", padrao={"meta_ativa": None, "solicitacao_exclusao": None})
    meta = config.get("meta_ativa")
    if consumo_atual_m3 is None:
        try:
            dados = carregar_json(ARQUIVO_HISTORICO)
            consumo_atual_m3 = float(dados[-1].get("consumo_m3", 0.0))
        except Exception:
            consumo_atual_m3 = 0.0
    if not meta:
        relatorio = {
            "meta_ativa": False,
            "mensagem": "Nenhuma meta ativa configurada.",
            "consumo_atual_m3": consumo_atual_m3
        }
    else:
        limite = float(meta.get("valor", 1.0))
        porcentagem = round((consumo_atual_m3 / limite) * 100, 2)
        saldo_restante = round(limite - consumo_atual_m3, 2)
        if porcentagem >= 100:
            situacao = "ESTOURADA"
        elif porcentagem >= 80:
            situacao = "ALERTA_80"
        else:
            situacao = "NORMAL"
        relatorio = {
            "meta_ativa": True,
            "meta_nome": meta.get("nome", "Meta de Consumo"),
            "limite_m3": limite,
            "consumo_atual_m3": consumo_atual_m3,
            "percentual_atingido": porcentagem,
            "saldo_restante_m3": saldo_restante,
            "situacao": situacao
        }
    if salvar:
        salvar_json("progresso_meta.json", relatorio)
    return relatorio


In [ ]:
def executar_controlador(estado_solicitado: Any = None) -> Dict[str, Any]:
    contexto = carregar_contexto()
    estado = estado_solicitado or contexto.get("estado") or contexto.get("acao") or ESTADO
    resultado_execucao = {
        "estado_executado": estado,
        "timestamp": datetime.now().isoformat(),
        "outputs_gerados": []
    }
    match str(estado).lower().strip():
        case "1" | "ml":
            resultado_ml = executar_pipeline_ml(salvar=True)
            resultado_execucao["ml"] = resultado_ml
            resultado_execucao["outputs_gerados"].append("previsao_ml.json")
        case "2" | "grafico":
            resultado_grafico = processar_dados_grafico(salvar=True)
            resultado_execucao["grafico"] = resultado_grafico
            resultado_execucao["outputs_gerados"].append("grafico_anual.json")
        case "3" | "metas":
            consumo_ctx = contexto.get("consumo_atual")
            resultado_meta = avaliar_progresso_meta(consumo_atual_m3=consumo_ctx, salvar=True)
            resultado_execucao["metas"] = resultado_meta
            resultado_execucao["outputs_gerados"].append("progresso_meta.json")
        case _:
            resultado_grafico = processar_dados_grafico(salvar=True)
            resultado_execucao["grafico"] = resultado_grafico
            resultado_execucao["outputs_gerados"].append("grafico_anual.json")
            resultado_ml = executar_pipeline_ml(salvar=True)
            resultado_execucao["ml"] = resultado_ml
            resultado_execucao["outputs_gerados"].append("previsao_ml.json")
            meses_validos = [m["consumo_m3"] for m in resultado_grafico["meses"] if m["consumo_m3"] is not None]
            consumo_recente = meses_validos[-1] if meses_validos else 0.0
            resultado_meta = avaliar_progresso_meta(consumo_atual_m3=float(consumo_recente), salvar=True)
            resultado_execucao["metas"] = resultado_meta
            resultado_execucao["outputs_gerados"].append("progresso_meta.json")
            sumario = {
                "status": "sucesso",
                "previsao_proximo_mes_m3": resultado_ml.get("previsao_m3"),
                "fatura_estimada_reais": resultado_ml.get("valor_estimado_conta"),
                "ano_grafico": resultado_grafico.get("ano_referencia"),
                "meta_status": resultado_meta.get("situacao") if resultado_meta else "sem_meta_ativa"
            }
            salvar_json("sumario_execucao.json", sumario)
            resultado_execucao["outputs_gerados"].append("sumario_execucao.json")
    return resultado_execucao


In [ ]:
resultado = executar_controlador()
dir_saida = obter_diretorio_output()
arquivos = list(dir_saida.glob("*.json"))
print(f"Execucao concluida. Arquivos no OUTPUT_DIR ({dir_saida}):")
for a in arquivos:
    print(f" - {a.name} ({a.stat().st_size} bytes)")
